# Modelo 8 — Isolation Forest

## 1. Dados

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, IsolationForest
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    fbeta_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.preprocessing import QuantileTransformer

raiz = Path.cwd()
if not (raiz / 'models' / 'modelo-5' / 'painel.parquet').exists():
    for pasta in Path.cwd().parents:
        candidato = pasta / 'models' / 'modelo-5' / 'painel.parquet'
        if candidato.exists():
            raiz = pasta
            break

painel = pd.read_parquet(raiz / 'models' / 'modelo-5' / 'painel.parquet')
painel['Data'] = pd.to_datetime(painel['Data'])

## 2. Features

In [2]:
PROIBIDAS = {'n_falhas_7d_futuro'}
EPS = 1.0

for prefixo in ('n_total', 'n_LSD', 'n_tecnicos', 'n_long30', 'n_FO', 'n_FC', 'n_RCE'):
    painel[f'rampa_{prefixo}'] = painel[f'{prefixo}_7d'] - (painel[f'{prefixo}_14d'] - painel[f'{prefixo}_7d'])

painel['razao_7_30'] = painel['n_total_7d'] / (painel['n_total_30d'] / 4 + EPS)
painel['razao_lsd_7_30'] = painel['n_LSD_7d'] / (painel['n_LSD_30d'] / 4 + EPS)
painel['razao_tec_7_30'] = painel['n_tecnicos_7d'] / (painel['n_tecnicos_30d'] / 4 + EPS)
painel['frac_tec_7'] = painel['n_tecnicos_7d'] / (painel['n_total_7d'] + EPS)
painel['frac_lsd_7'] = painel['n_LSD_7d'] / (painel['n_tecnicos_7d'] + EPS)
painel['frac_long_7'] = painel['n_long30_7d'] / (painel['n_total_7d'] + EPS)
painel['frac_fo_7'] = painel['n_FO_7d'] / (painel['n_tecnicos_7d'] + EPS)
painel['intensidade_7'] = np.log1p(painel['dur_max_7d']) / (painel['n_total_7d'] + EPS)
painel['lsd_x_dur'] = painel['n_LSD_7d'] * np.log1p(painel['dur_max_7d'])
painel['tec_x_dur'] = painel['n_tecnicos_7d'] * np.log1p(painel['dur_max_7d'])
painel['frescor_total'] = painel['n_total_decaido_h3'] / (painel['n_total_decaido_h14'] + EPS)
painel['frescor_lsd'] = painel['n_LSD_decaido_h7'] / (painel['n_LSD_7d'] + EPS)

tipos_7d = ['n_CA_7d', 'n_CSD_7d', 'n_FC_7d', 'n_FO_7d', 'n_LSD_7d', 'n_OA_7d', 'n_OBS_7d', 'n_RCE_7d']
painel['n_tipos_7d'] = (painel[tipos_7d] > 0).sum(axis=1)

for origem, dest in (
    ('dias_desde_zm01', 'rec_zm01'),
    ('dias_desde_zm02', 'rec_zm02'),
    ('dias_desde_zm03', 'rec_zm03'),
):
    dias = painel[origem].replace(9999, np.nan).clip(upper=365).fillna(365)
    painel[dest] = np.exp(-dias / 30.0)

painel['rec_ultima_corretiva'] = np.maximum(painel['rec_zm01'], painel['rec_zm02'])
painel['zm01_concentracao'] = painel['zm01_30d'] / (painel['zm01_90d'] / 3 + EPS)
dias_z = painel['dias_desde_zm01'].replace(9999, 365).clip(0, 365)
painel['razao_atraso'] = dias_z / (90.0 / painel['zm01_90d'].clip(lower=0.5))

for col in ('n_tecnicos_7d', 'n_LSD_7d', 'dur_max_7d', 'n_long30_7d'):
    g = painel.groupby(['est', 'Data'])[col]
    med = g.transform('median')
    std = g.transform('std').replace(0, np.nan).fillna(1.0)
    painel[f'zest_{col}'] = ((painel[col] - med) / std).clip(-8, 8)

id_cols = {
    'id_porta', 'Data', 'est', 'via', 'id_plat', 'y', 'cobertura_30d',
    'mes', 'dia_semana', 'fim_de_semana', 'num_porta',
}
candidatas = [
    c for c in painel.columns
    if c not in PROIBIDAS and c not in id_cols
    and not c.startswith('dias_desde_')
    and not c.endswith(('_1d', '_3d'))
    and pd.api.types.is_numeric_dtype(painel[c])
]
print(len(candidatas), 'candidatas')

76 candidatas


## 3. Split e seleção

In [3]:
FIM_FIT = pd.Timestamp('2025-01-23')
INI_VALID = pd.Timestamp('2025-01-31')
FIM_VALID = pd.Timestamp('2025-04-23')
INI_TESTE = pd.Timestamp('2025-05-01')

usar = ['id_porta', 'Data', 'est', 'via', 'num_porta', 'y'] + candidatas
painel = painel.loc[:, usar]
mask_tr = painel['Data'] <= FIM_FIT
prior_porta = painel.loc[mask_tr].groupby('id_porta')['y'].mean()
prior_est = painel.loc[mask_tr].groupby('est')['y'].mean()
med_est = float(prior_est.median())
painel['prior_porta'] = painel['id_porta'].map(prior_porta).fillna(med_est)
painel['prior_est'] = painel['est'].map(prior_est).fillna(med_est)
painel['via_n'] = pd.to_numeric(painel['via'], errors='coerce').fillna(0)
painel['num_porta_n'] = pd.to_numeric(painel['num_porta'], errors='coerce').fillna(0)

treino = painel.loc[mask_tr]
valid = painel.loc[painel['Data'].between(INI_VALID, FIM_VALID)]
teste = painel.loc[painel['Data'] >= INI_TESTE]
del painel

y_tr = treino['y'].astype(int).to_numpy()
y_va = valid['y'].astype(int).to_numpy()
y_te = teste['y'].astype(int).to_numpy()
base_ap = float(y_tr.mean())


def ap_univariada(serie, y):
    v = pd.to_numeric(serie, errors='coerce').replace([np.inf, -np.inf], np.nan)
    v = v.fillna(v.median() if v.notna().any() else 0.0)
    return max(average_precision_score(y, v), average_precision_score(y, -v))


ranking = {c: ap_univariada(treino[c], y_tr) for c in candidatas}
ordem = sorted(ranking, key=ranking.get, reverse=True)
feats = [c for c in ordem if ranking[c] >= base_ap * 1.10][:36]
if len(feats) < 18:
    feats = ordem[:22]

X_corr = treino[feats].replace([np.inf, -np.inf], np.nan).fillna(0)
corr = X_corr.corr().abs()
kept = []
for c in feats:
    if X_corr[c].std() < 1e-9:
        continue
    if all(corr.loc[c, k] < 0.94 for k in kept):
        kept.append(c)
feats = kept
print(len(feats), 'features')
print(*[f'{c}:{ranking[c]:.3f}' for c in feats[:12]], sep=' | ')

20 features
rec_zm01:0.120 | zm01_90d:0.119 | zm01_30d:0.108 | frac_lsd_7:0.103 | frac_tec_7:0.101 | zm01_concentracao:0.100 | razao_atraso:0.098 | frac_fo_7:0.093 | zest_n_tecnicos_7d:0.093 | zest_n_LSD_7d:0.090 | n_total_decaido_h3:0.088 | n_tecnicos_7d:0.088


## 4. Isolation Forest

In [4]:
def limpar(df):
    X = df[feats].replace([np.inf, -np.inf], np.nan).astype(float)
    for c in X.columns:
        if c.startswith('zest_'):
            X[c] = X[c].clip(-8, 8)
        elif (X[c].dropna() >= -1e-12).all():
            X[c] = np.log1p(X[c].clip(lower=0))
    return X


def residual(X, ids, ref):
    base = ref.reindex(ids)
    base.index = X.index
    return X.sub(base.fillna(0)).fillna(0)


normais = treino[treino['y'] == 0]
lim_tec = normais['n_tecnicos_7d'].quantile(0.70)
fit_if = normais[(normais['n_tecnicos_7d'] <= lim_tec) & (normais['zm01_30d'] <= 0)].copy()
if len(fit_if) < 8000:
    fit_if = normais[normais['n_tecnicos_7d'] <= normais['n_tecnicos_7d'].quantile(0.80)].copy()

X_fit = limpar(fit_if)
ref_porta = X_fit.groupby(fit_if['id_porta'].values).median()
X_fit = residual(X_fit, fit_if['id_porta'].values, ref_porta)
lo, hi = X_fit.quantile(0.005), X_fit.quantile(0.995)
X_fit = X_fit.clip(lower=lo, upper=hi, axis=1)

scaler = QuantileTransformer(
    output_distribution='normal',
    n_quantiles=min(1000, len(X_fit)),
    subsample=min(80_000, len(X_fit)),
    random_state=42,
)
X_fit_s = scaler.fit_transform(X_fit)

iso = IsolationForest(
    n_estimators=280,
    max_samples=min(2048, len(X_fit_s)),
    max_features=min(1.0, max(0.6, 10 / max(len(feats), 1))),
    contamination='auto',
    random_state=42,
    n_jobs=-1,
)
iso.fit(X_fit_s)


def score_if(df):
    X = residual(limpar(df), df['id_porta'].values, ref_porta)
    Xs = scaler.transform(X.clip(lower=lo, upper=hi, axis=1).fillna(0))
    return -iso.score_samples(Xs)


def bloco(df):
    X = df[feats + ['prior_porta', 'prior_est', 'via_n', 'num_porta_n']].replace(
        [np.inf, -np.inf], np.nan
    ).astype(float)
    X['iso'] = score_if(df)
    return X


X_tr, X_va, X_te = bloco(treino), bloco(valid), bloco(teste)
print('IF em', len(fit_if), 'normais')

IF em 100971 normais


## 5. Calibração e métricas

In [ ]:
clf = HistGradientBoostingClassifier(
    max_depth=4,
    min_samples_leaf=90,
    max_iter=160,
    learning_rate=0.07,
    l2_regularization=1.2,
    class_weight='balanced',
    early_stopping=True,
    validation_fraction=0.12,
    n_iter_no_change=20,
    random_state=42,
)
clf.fit(X_tr, y_tr)

score_va = clf.predict_proba(X_va)[:, 1]
score_te = clf.predict_proba(X_te)[:, 1]

melhor = (-1.0, 0.5)
for t in np.unique(np.concatenate([
    np.quantile(score_va, np.linspace(0.35, 0.92, 50)),
    np.linspace(0.20, 0.75, 56),
])):
    f2 = fbeta_score(y_va, score_va >= t, beta=2, zero_division=0)
    if f2 > melhor[0]:
        melhor = (f2, float(t))

f2_va, limiar = melhor
pred_te = (score_te >= limiar).astype(int)

print('limiar:', round(limiar, 4), '| F2 val:', round(f2_va, 3), '| iter:', clf.n_iter_)
print('PR-AUC teste:', round(average_precision_score(y_te, score_te), 4),
      '| baseline:', round(float(y_te.mean()), 4))
print('classe 1  P:', round(precision_score(y_te, pred_te, zero_division=0), 3),
      'R:', round(recall_score(y_te, pred_te, zero_division=0), 3),
      'F2:', round(fbeta_score(y_te, pred_te, beta=2, zero_division=0), 3))
print(classification_report(y_te, pred_te, digits=3, zero_division=0))

## 6. Avaliação por evento

In [ ]:
def n_inicios(df, col):
    s = df.sort_values(['id_porta', 'Data'])
    flag = s[col].astype(bool)
    prev = flag.groupby(s['id_porta']).shift(1).fillna(False).astype(bool)
    return int((flag & ~prev).sum())


def eventos_pegos(df, y_col, pred_col):
    s = df.sort_values(['id_porta', 'Data']).copy()
    y = s[y_col].astype(bool)
    novo = y & ~y.groupby(s['id_porta']).shift(1).fillna(False).astype(bool)
    s['_eid'] = novo.groupby(s['id_porta']).cumsum()
    falhas = s[y]
    if falhas.empty:
        return 0, 0
    grupos = falhas.groupby(['id_porta', '_eid'], sort=False)
    return int(grupos[pred_col].max().sum()), int(grupos.ngroups)


avaliar = teste.copy()
avaliar['pred'] = pred_te
avaliar['y'] = y_te
pegos, n_eventos = eventos_pegos(avaliar, 'y', 'pred')
n_avisos = n_inicios(avaliar, 'pred')
avisos_ok, _ = eventos_pegos(avaliar, 'pred', 'y')
print('eventos', n_eventos, 'pegos', pegos, 'recall evento', round(pegos / n_eventos, 3) if n_eventos else 0)
print('avisos', n_avisos, 'acertaram', avisos_ok, 'precisão evento', round(avisos_ok / n_avisos, 3) if n_avisos else 0)

## 7. Curva Precision-Recall

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4))
p, r, _ = precision_recall_curve(y_te, score_te)
ax.plot(r, p, label=f'PR-AUC {average_precision_score(y_te, score_te):.3f}')
ax.axhline(y_te.mean(), color='gray', linestyle='--', label=f'baseline {y_te.mean():.3f}')
ax.set_xlabel('Recall')
ax.set_ylabel('Precisão')
ax.legend()
fig.tight_layout()
plt.show()